In [10]:
import pandas as pd
import numpy as np

In [2]:
import scipy.stats as st
from scipy.fftpack import fft, fftfreq 
from scipy.signal import argrelextrema
import operator

In [3]:
def stat_area_features(x, Te=1.0):

    mean_ts = np.mean(x, axis=1).reshape(-1,1) # mean
    max_ts = np.amax(x, axis=1).reshape(-1,1) # max
    min_ts = np.amin(x, axis=1).reshape(-1,1) # min
    std_ts = np.std(x, axis=1).reshape(-1,1) # std
    skew_ts = st.skew(x, axis=1).reshape(-1,1) # skew
    kurtosis_ts = st.kurtosis(x, axis=1).reshape(-1,1) # kurtosis 
    iqr_ts = st.iqr(x, axis=1).reshape(-1,1) # interquartile rante
    mad_ts = np.median(np.sort(abs(x - np.median(x, axis=1).reshape(-1,1)),
                               axis=1), axis=1).reshape(-1,1) # median absolute deviation
    area_ts = np.trapz(x, axis=1, dx=Te).reshape(-1,1) # area under curve
    sq_area_ts = np.trapz(x ** 2, axis=1, dx=Te).reshape(-1,1) # area under curve ** 2

    return np.concatenate((mean_ts,max_ts,min_ts,std_ts,skew_ts,kurtosis_ts,
                           iqr_ts,mad_ts,area_ts,sq_area_ts), axis=1)

In [4]:
def frequency_domain_features(x, Te=1.0):

    # As the DFT coefficients and their corresponding frequencies are symetrical arrays
    # with respect to the middle of the array we need to know if the number of readings 
    # in x is even or odd to then split the arrays...
    if x.shape[1]%2 == 0:
        N = int(x.shape[1]/2)
    else:
        N = int(x.shape[1]/2) - 1
    xf = np.repeat(fftfreq(x.shape[1],d=Te)[:N].reshape(1,-1), x.shape[0], axis=0) # frequencies
    dft = np.abs(fft(x, axis=1))[:,:N] # DFT coefficients   
    
    # statistical and area features
    dft_features = stat_area_features(dft, Te=1.0)
    # weighted mean frequency
    dft_weighted_mean_f = np.average(xf, axis=1, weights=dft).reshape(-1,1)
    # 5 first DFT coefficients 
    dft_first_coef = dft[:,:5]    
    # 5 first local maxima of DFT coefficients and their corresponding frequencies
    dft_max_coef = np.zeros((x.shape[0],5))
    dft_max_coef_f = np.zeros((x.shape[0],5))
    for row in range(x.shape[0]):
        # finds all local maximas indexes
        extrema_ind = argrelextrema(dft[row,:], np.greater, axis=0) 
        # makes a list of tuples (DFT_i, f_i) of all the local maxima
        # and keeps the 5 biggest...
        extrema_row = sorted([(dft[row,:][j],xf[row,j]) for j in extrema_ind[0]],
                             key=operator.itemgetter(0), reverse=True)[:5] 
        for i, ext in enumerate(extrema_row):
            dft_max_coef[row,i] = ext[0]
            dft_max_coef_f[row,i] = ext[1]    
    
    return np.concatenate((dft_features,dft_weighted_mean_f,dft_first_coef,
                           dft_max_coef,dft_max_coef_f), axis=1)


In [6]:
def convert_to_list(df,column_name):
    data=df
    data['timestamp'] = pd.to_datetime(data['timestamp'])
    data['timestamp'] = data['timestamp'].dt.floor(freq='4H')
    Unique_time = data.timestamp.unique()
    X = []
    column_name=column_name
    for time in Unique_time:
        values = data[data.timestamp==time][column_name].tolist()
        if len(values)<721:
            resample_len=721-len(values)
            new_data = np.random.choice(values, resample_len)
            values.extend(new_data)
        X.append(values)
    return np.array(X)

In [7]:
def make_feature(df,column_name):
    data=df
    column_name=column_name
    x=convert_to_list(data,column_name=column_name)
    x_stat=stat_area_features(x)
    x_stat=pd.DataFrame(x_stat)
    var = column_name
    column1=[var+'_stat_'+ str(I) for I in range(x_stat.shape[1])]
    x_stat.columns=column1
    x_domain=frequency_domain_features(x)
    x_domain=pd.DataFrame(x_domain)
    column2=[var+'_tm_dom_'+ str(I) for I in range(x_domain.shape[1])]
    x_domain.columns=column2
    x_final=pd.concat([x_stat, x_domain], axis=1)
    return x_final

In [8]:
data=pd.read_csv('database_gas.csv')

In [11]:
humidity=make_feature(data,column_name='humidity')

In [12]:
humidity

,humidity_stat_0,humidity_stat_1,humidity_stat_2,humidity_stat_3,humidity_stat_4,humidity_stat_5,humidity_stat_6,humidity_stat_7,humidity_stat_8,humidity_stat_9,...,humidity_tm_dom_16,humidity_tm_dom_17,humidity_tm_dom_18,humidity_tm_dom_19,humidity_tm_dom_20,humidity_tm_dom_21,humidity_tm_dom_22,humidity_tm_dom_23,humidity_tm_dom_24,humidity_tm_dom_25
0,51.919209,54.86,47.84,2.500421,-0.468320,-1.572980,4.80,1.15,37379.325,1.945076e+06,...,229.536738,161.113400,160.782758,148.617904,146.072604,0.008322,0.134535,0.337032,0.488211,0.019417
1,50.238932,53.03,46.07,1.352963,-0.177450,-0.281509,1.80,0.90,36173.305,1.818690e+06,...,371.317543,192.117632,148.763596,142.459793,95.090692,0.002774,0.006935,0.009709,0.023578,0.015257
2,51.342122,54.72,46.62,1.828906,-0.248747,-0.703490,2.89,1.41,36965.160,1.900220e+06,...,149.790040,146.068927,100.018081,83.339716,62.134019,0.008322,0.012483,0.015257,0.019417,0.031900
3,51.510957,56.37,46.34,3.167119,0.030124,-1.428285,6.02,2.99,37091.480,1.918020e+06,...,645.262261,403.906696,198.378101,171.888541,138.812969,0.004161,0.006935,0.011096,0.019417,0.022191
4,56.478890,57.17,55.44,0.389103,-0.677984,-0.174461,0.46,0.24,40664.780,2.296810e+06,...,61.350935,60.230271,24.867698,24.189503,23.388714,0.004161,0.008322,0.015257,0.026352,0.012483
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
577,58.628599,59.21,57.32,0.392339,-0.818312,0.305726,0.59,0.28,42212.910,2.475013e+06,...,116.767797,65.268793,64.483112,52.685119,39.920378,0.002774,0.009709,0.012483,0.005548,0.015257
578,58.629958,59.30,57.17,0.428274,-0.964831,0.966131,0.51,0.27,42213.710,2.475128e+06,...,74.047005,57.477791,53.696660,39.471130,38.635596,0.009709,0.012483,0.015257,0.024965,0.018031
579,58.582899,59.14,57.48,0.305120,-1.002789,1.564696,0.32,0.16,42179.325,2.471033e+06,...,64.931082,53.889661,48.770261,46.217101,45.424453,0.006935,0.002774,0.009709,0.015257,0.012483
580,58.429043,59.17,57.51,0.370834,-0.594740,-0.285363,0.48,0.24,42069.355,2.458197e+06,...,116.485255,67.920980,54.249328,45.197844,43.342139,0.002774,0.009709,0.006935,0.016644,0.013870
